 # Protostellar Outflows DATA Pipeline

This project aims to retrieve ALMA observations of protostellar outflows from the ALMA Science Archive using TAP queries and build a sample of sources exhibiting episodic outflow morphology. Our initial goal is to identify observations of the SiO (8–7) molecular line, which traces shocked gas associated with high-velocity protostellar jets and outflows. Once we have established a sample of sources with SiO (8–7) observations, we will expand our search to include the CO (3–2) and CO (2–1) transitions to further investigate the morphology of protostellar outflows.

# These commands are helpful to see astroquery version and relevant steuptools


In [1]:
#import pkg_resources
#print(pkg_resources.__file__)
#pip install setuptools
#import sys
#print(sys.executable)
#print(sys.version)

# Import Libraries

In [51]:
from astroquery.alma import Alma
import os
import numpy as np
import pandas as pd
from astroquery.alma.utils import parse_frequency_support
import astropy.units as u
from astropy.table import Table
import shutil
from pathlib import Path
import tarfile
from astropy.io import fits
from spectral_cube import SpectralCube
import matplotlib.pyplot as plt 
#import alminer
from astropy.wcs import WCS
from astropy.visualization.wcsaxes import add_beam
from glob import glob
from urllib.parse import urlparse, unquote

# Search for relevant molecular line

In [52]:
#Add sio rest frequency
sio_rest = 347.330581 * u.GHz # From Splatalouge SiO (8-7)

#CO_rest = 345.7959899 * u.GHz #CO (3-2)
# Frequency search range
freq_min = 347.0  # GHz
freq_max = 347.7  # GHz

#Speed of light
c = 299792458.0


lambda_min = c / (freq_max * 1e9)
lambda_max = c / (freq_min * 1e9)


In [53]:
# Connection to the ALMA Archive
alma = Alma()
alma.archive_url = 'https://almascience.eso.org'

# Use ALMA AstroTAP Queries to Identify Data Products Based on Science Keywords in the ALMA Science Archive

In [6]:

query = """
SELECT
    obs_publisher_did,
    member_ous_uid,
    target_name,
    frequency_support,
    science_keyword,
    antenna_arrays
FROM ivoa.obscore
WHERE dataproduct_type = 'cube'
AND (
      science_keyword LIKE '% jets and ionized winds%'
      OR science_keyword LIKE '%disks around low mass stars%'
  )
AND (
      antenna_arrays LIKE '%:DA%'
      OR antenna_arrays LIKE '%:DV%'
      OR antenna_arrays LIKE '%:PM%'
  )
AND antenna_arrays NOT LIKE '%:CM%'
"""

#Print results for total observations found with those keywords
results = alma.query_tap(query)

print("Total 12m observations:", len(results))


Total 12m observations: 20624


# Parse the frequency support in the ALMA Data

In [74]:
# Frequency support is stored as a string, so we need to parse it to extract only the frequency ranges
frequency_support_str = results['frequency_support'][:1]
print(frequency_support_str)
print(type(frequency_support_str))

['[345.55..345.67GHz,141.11kHz,28.6mJy/beam@10km/s,8.7mJy/beam@native, XX YY] U [345.73..345.86GHz,141.11kHz,28.3mJy/beam@10km/s,8.6mJy/beam@native, XX YY] U [346.94..347.06GHz,141.11kHz,27.2mJy/beam@10km/s,8.3mJy/beam@native, XX YY] U [347.27..347.39GHz,141.11kHz,27.1mJy/beam@10km/s,8.3mJy/beam@native, XX YY] U [356.05..358.05GHz,31250.00kHz,33.2mJy/beam@10km/s,2.6mJy/beam@native, XX YY] U [356.67..356.80GHz,121.15kHz,32.2mJy/beam@10km/s,9.9mJy/beam@native, XX YY]']
<class 'numpy.ma.MaskedArray'>


In [75]:
#Parse the frequncy support
for a in range(len(frequency_support_str)):
    freq_ranges = parse_frequency_support(frequency_support_str[a])
    print(freq_ranges,a)
    

[[345.55 345.67]
 [345.73 345.86]
 [346.94 347.06]
 [347.27 347.39]
 [356.05 358.05]
 [356.67 356.8 ]] GHz 0


# Define a frequency range to identify relevant data products that cover the molecular line's rest frequency.

In [76]:
# Set up frequency range and in results find the range

matches = [] #Empty list
#print(type(results))
for ind,x in enumerate(results):
    #if ind >= 2:
        #continue
    ranges = parse_frequency_support(x['frequency_support'])
    #print(ranges)

    for fmin, fmax in ranges:
        if sio_rest >= fmin and sio_rest <= fmax:
            matches.append(x)
            break
print(len(matches))

244


In [77]:
#convert results to a table
table = results.to_table()

In [78]:
#print columnames
table.colnames

['obs_publisher_did',
 'member_ous_uid',
 'target_name',
 'frequency_support',
 'frequency',
 'bandwidth',
 'science_keyword',
 'antenna_arrays']

# Matches Found Within the Specified Frequency Range

In [79]:
# Convert list to a astropy.table
matches = Table(rows=matches, names=table.colnames)
#print(len(matches))

In [80]:
#print(matches)

In [81]:
#subset = 0 # only for 1 source
subset = matches[:] #download all files

In [82]:
#print(len(subset))
#print(subset['member_ous_uid'])

# Expand files information

In [ ]:

uid = subset['member_ous_uid']# 600 rows from the sample

# get products information and expand tarfiles
products = alma.get_data_info (uid,expand_tarfiles=True)

#print(products)

# Select only TAR files containing science data products

In [ ]:

science_tar = products[
    [('001_of_001.tar' in str(x)) for x in products['access_url']]
]

#print(len(science_tar))
#print(science_tar)

# Extract TAR files and select only spectral cube data products

In [71]:
cube_files = []

for row in products:
    if 'cube' in str(row).lower() and '.fits' in str(row).lower():
        cube_files.append(row)
        print(row)

for x in cube_files:
    print(len(x))

         ID                                                                       access_url                                                             service_def error_message semantics local_semantics                        description                           content_type   content_length content_qualifier link_auth link_authorized
                                                                                                                                                                                                                                                                                              byte                                                 
-------------------- ----------------------------------------------------------------------------------------------------------------------------------- ----------- ------------- --------- --------------- ---------------------------------------------------------- ---------------- -------------- ----------------- ------

# Check the scratch_files directory for previously downloaded files, skip existing files, and download only the missing data products to avoid duplicate downloads.

In [25]:
alma.cache_location = '/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files'

download_dir = Path("/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files")
download_dir.mkdir(parents=True, exist_ok=True)

# Check which files are already exist
files_to_download = []

for row in cube_files:

    url = str(row['access_url'])

    # Extract filename from URL
    filename = Path(unquote(urlparse(url).path)).name

    destination = download_dir / filename

    if destination.exists() and destination.stat().st_size > 0:
        print("Already downloaded, skipping:", filename)
    else:
        files_to_download.append(url)

print("Files to download:", len(files_to_download))

# Download only missing files
downloaded_files = []

if files_to_download:
    downloaded_files = alma.download_files(files_to_download)

# Move downloaded files
for f in downloaded_files:

    source = Path(f)
    destination = download_dir / source.name

    if source.resolve() != destination.resolve():
        shutil.move(str(source), str(destination))

print("Download process completed!")


Already downloaded, skipping: member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw1_349001MHz.12m.cube.I.pb.fits.gz
Already downloaded, skipping: member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw1_349001MHz.12m.cube.I.pbcor.fits
Already downloaded, skipping: member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw2_337393MHz.12m.cube.I.mask.fits.gz
Already downloaded, skipping: member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw2_337393MHz.12m.cube.I.pb.fits.gz
Already downloaded, skipping: member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw2_337393MHz.12m.cube.I.pbcor.fits
Already downloaded, skipping: member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw3_335485MHz.12m.cube.I.pb.fits.gz
Already downloaded, skipping: member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw3_335485MHz.12m.cube.I.pbcor.fits
Already downloaded, skipping: member.uid___A001_X122_X4a1.ari_l.J1517-2422_bp.spw0_347329MHz.12m.cube.I.pb.fits.gz
Already downloaded, skipping: member

In [26]:

from pathlib import Path

# Directory containing previously downloaded FITS files
download_dir = Path("/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files")

# Find existing FITS files
downloaded_files = sorted(download_dir.glob("*.fits"))

print("Total existing FITS files:", len(downloaded_files))

for f in downloaded_files:
    print(f.name)


Total existing FITS files: 215
member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw0_347327MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw1_349001MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw2_337393MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw3_335485MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a1.ari_l.J1517-2422_bp.spw0_347329MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a1.ari_l.J1517-2422_bp.spw1_349004MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a1.ari_l.J1517-2422_bp.spw2_337395MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a1.ari_l.J1517-2422_bp.spw3_335487MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a1.ari_l.J1625-2527_ph.spw0_347328MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a1.ari_l.J1625-2527_ph.spw1_349002MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a1.ari_l.J1625-2527_ph.spw2_337394MHz.12m.cube.I.pbcor.fits
member.uid

In [1]:
#print(len(downloaded_files))

In [37]:
#print(downloaded_files)

In [38]:
#print(download_dir.resolve())
#print(alma.cache_location)

In [39]:
#print(download_dir.exists())
#p = Path("/users/skaur/Machine-Learning-Project/Astromorph_project/alma_astroquery/alma_TAP/scratch_files")
#print(p.exists())

In [40]:
#Alma.help_tap()

# Get Rest Frequency for all fits files

In [27]:
#download_dir = Path('/users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files')

for f in download_dir.glob("*.fits"):
    if "cube" in f.name.lower():
        header = fits.getheader(f)

        print("\n", f.name)
        print("Rest frequency:", header.get("RESTFRQ", "No RESTFRQ"))


 member.uid___A001_X3621_X98a.DHNb71_sci.spw21.cube.I.sd.fits
Rest frequency: 347330578600.0

 member.uid___A001_X1465_X1bc.Lh06_sci.spw29.cube.I.pbcor.fits
Rest frequency: 347060000000.0

 member.uid___A001_X1465_X1bc.Lh03_sci.spw27.cube.I.pbcor.fits
Rest frequency: 345650000000.0

 member.uid___A001_X2d20_Xebd._G206.93-16.61W2__sci.spw25.cube.I.pbcor.fits
Rest frequency: 346528481000.0

 member.uid___A001_X1465_X2b6b.HH_211_sci.spw29.cube.I.pbcor.fits
Rest frequency: 346998347000.0

 member.uid___A001_X3788_X5e32.10295-5746_sci.spw31.cube.I.selfcal.pbcor.fits
Rest frequency: 358605800000.0

 member.uid___A001_X133d_X3cbd.iras16293-2422_sci.spw17.cube.I.sd.fits
Rest frequency: 345795990000.0

 member.uid___A001_X1465_X1c0.S03_sci.spw29.cube.I.pbcor.fits
Rest frequency: 347060000000.0

 member.uid___A001_X1465_X1c0.S09_sci.spw25.cube.I.pbcor.fits
Rest frequency: 344660000000.0

 member.uid___A001_X1465_X1bc.Lh08_sci.spw27.cube.I.pbcor.fits
Rest frequency: 345650000000.0

 member.uid__

# Use the SiO (8–7) rest frequency to identify the corresponding SiO FITS files in the scratch_files directory.

In [28]:
# Use rest frequency to find the right SiO cube FITS files
sio_files = []

for f in download_dir.glob("*.fits"):

    header = fits.getheader(f)

    restfreq = header.get("RESTFRQ", header.get("RESTFREQ", None))

    if restfreq is not None:

        freq1 = header["CRVAL3"] + (1 - header["CRPIX3"]) * header["CDELT3"]
        freq2 = header["CRVAL3"] + (header["NAXIS3"] - header["CRPIX3"]) * header["CDELT3"]

        freq_min_GHz = min(freq1, freq2) / 1e9
        freq_max_GHz = max(freq1, freq2) / 1e9

        if freq_min_GHz <= 347.33058 <= freq_max_GHz:
            sio_files.append(f)

In [29]:
print("SiO files:")

for f in sio_files:
    print(f)

print("Total SiO files:", len(sio_files))

SiO files:
/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X3621_X98a.DHNb71_sci.spw21.cube.I.sd.fits
/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1bc.Lh06_sci.spw29.cube.I.pbcor.fits
/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1c0.S03_sci.spw29.cube.I.pbcor.fits
/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X3788_X5e32.10295-5746_sci.spw27.cube.I.selfcal.pbcor.fits
/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X133d_X3cb7.iras16293-2422_sci.spw19.cube.I.sd.fits
/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1bc.Lh03_sci.spw29.cube.I.pbcor.fits
/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A0

In [30]:
# Check all files in the scratch_directory
all_files = glob.glob("/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/*.fits*")
#all_files
#print(Path(all_files[0]).name)

# Keep the SiO FITS files and their corresponding PB, PBCOR, mask, and self-calibrated files for each SiO spectral window

In [31]:

keep_files = []

# Get observation IDs and SPWs from SiO fits files
sio_spws = set()

for f in sio_files:
    filename = Path(f).name.lower()

    match = re.search(r"(member\.uid.*?_sci)\.(spw\d+)", filename)

    if match:
        sio_spws.add((match.group(1), match.group(2)))

# Keep corresponding files
for var in all_files:
    filename = Path(var).name.lower()

    match = re.search(r"(member\.uid.*?_sci)\.(spw\d+)", filename)

    if match and (match.group(1), match.group(2)) in sio_spws:

        if (".pb." in filename or
            ".pbcor." in filename or
            ".mask." in filename or
            ("selfcal" in filename and "pbcor" in filename)):

            keep_files.append(var)

print("SiO observation and spectral windows:", sio_spws)
print("Total keep files:", len(keep_files))

SiO observation and spectral windows: {('member.uid___a001_x133d_x3cb7.iras16293-2422_sci', 'spw19'), ('member.uid___a001_x2d20_xebd._g206.93-16.61w2__sci', 'spw29'), ('member.uid___a001_x133d_x3cab.iras16293-2422_sci', 'spw19'), ('member.uid___a001_x1465_x1c0.s10_sci', 'spw29'), ('member.uid___a001_x3621_x990.dhnb72_sci', 'spw21'), ('member.uid___a001_x1465_x1c0.s04_sci', 'spw29'), ('member.uid___a001_x1465_x1c0.s02_sci', 'spw29'), ('member.uid___a001_x1465_x1bc.lh04_sci', 'spw29'), ('member.uid___a001_x1465_x1bc.lh01_sci', 'spw29'), ('member.uid___a001_x3788_x5e32.10295-5746_sci', 'spw27'), ('member.uid___a001_x1465_x1bc.lh05_sci', 'spw29'), ('member.uid___a001_x1465_x1c0.s05_sci', 'spw29'), ('member.uid___a001_x1465_x1bc.lh08_sci', 'spw29'), ('member.uid___a001_x1465_x1c0.s01_sci', 'spw29'), ('member.uid___a001_x1465_x1c0.s09_sci', 'spw29'), ('member.uid___a001_x1465_x2b6b.hh_211_sci', 'spw31'), ('member.uid___a001_x1465_x1c0.s07_sci', 'spw29'), ('member.uid___a001_x1465_x1bc.ll10_s

# Before deleting files without SiO emission, verify which files will be kept

In [32]:
keep_files_names = [Path(f).name for f in keep_files]

print("keep_files:")
for f in keep_files:
    print(repr(f))
    #print(len(f))


print("\nall_files:")
for f in all_files:
    print(repr(f))

keep_files:
'/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1bc.Lh06_sci.spw29.cube.I.pbcor.fits'
'/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1c0.S03_sci.spw29.cube.I.pb.fits.gz'
'/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1c0.S08_sci.spw29.cube.I.mask.fits.gz'
'/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1bc.Lm07_sci.spw29.cube.I.pb.fits.gz'
'/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1c0.S03_sci.spw29.cube.I.pbcor.fits'
'/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X3788_X5e32.10295-5746_sci.spw27.cube.I.selfcal.pbcor.fits'
'/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.u

# Move all selected SiO FITS files to the final_fits_file directory

In [33]:
for y in all_files:
    if Path(y).name in keep_files_names :
        destination = Path(y.replace('scratch_files', 'final_fits_file'))
        #print(destination)
        # Make sure destination directory exists
        destination.parent.mkdir(parents=True, exist_ok=True)

        shutil.move(Path(y), destination)
        print("Moved:", y)

Moved: /scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1bc.Lh06_sci.spw29.cube.I.pbcor.fits
Moved: /scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1c0.S03_sci.spw29.cube.I.pb.fits.gz
Moved: /scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1c0.S08_sci.spw29.cube.I.mask.fits.gz
Moved: /scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1bc.Lm07_sci.spw29.cube.I.pb.fits.gz
Moved: /scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1c0.S03_sci.spw29.cube.I.pbcor.fits
Moved: /scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X3788_X5e32.10295-5746_sci.spw27.cube.I.selfcal.pbcor.fits
Moved: /scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TA

# Delete FITS files that do not contain the SiO (8–7) spectral line

In [61]:
remaining_files = glob.glob('./alma_TAP/scratch_files/*')

for y in remaining_files:
    #os.remove(y)
    print("Deleted:", y)
print(len(remaining_files))

0
